In [144]:
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd


def load_trackmate_full_xml(path):
    """Parse a full TrackMate project XML and return spots and a merged metadata dict."""
    root = ET.parse(path).getroot()
    model = root.find('Model')

    spatial_units = model.attrib.get('spatialunits')
    time_units = model.attrib.get('timeunits')

    dim_to_unit = {
        'POSITION': spatial_units, 'LENGTH': spatial_units,
        'TIME': time_units, 'VELOCITY': f'{spatial_units}/{time_units}',
        'RATE': f'1/{time_units}', 'ANGLE_RATE': f'rad/{time_units}',
        'INTENSITY': 'counts', 'ANGLE': 'rad',
        'QUALITY': None, 'NONE': None, 'COST': None, 'STRING': None,
    }

    # feature -> unit, from <FeatureDeclarations>
    metadata = {}
    for category in model.find('FeatureDeclarations'):
        for feat in category:
            metadata[feat.attrib['feature']] = dim_to_unit.get(feat.attrib.get('dimension'))

    # spots
    spots = pd.DataFrame([
        spot.attrib
        for frame in model.find('AllSpots')
        for spot in frame
    ])
    spots['ID'] = spots['ID'].astype(np.int64)
    num_cols = [c for c in spots.columns if c != 'name']
    spots[num_cols] = spots[num_cols].apply(pd.to_numeric, errors='coerce')

    # tag each spot with its TRACK_ID via the edges (single pass, vectorized map)
    src_ids, trk_ids = [], []
    for track in model.find('AllTracks'):
        tid = int(track.attrib['TRACK_ID'])
        for edge in track:
            a = edge.attrib
            src_ids += [a['SPOT_SOURCE_ID'], a['SPOT_TARGET_ID']]
            trk_ids += [tid, tid]

    mapping = pd.Series(
        np.asarray(trk_ids, dtype=np.int64),
        index=np.asarray(src_ids, dtype=np.int64),
    )
    mapping = mapping[~mapping.index.duplicated()]
    spots['TRACK_ID'] = mapping.reindex(spots['ID'].values).values

    # move TRACK_ID to the first column
    spots.insert(0, 'TRACK_ID', spots.pop('TRACK_ID'))


    # merge image calibration into the same metadata dict
    metadata.update(root.find('Settings/ImageData').attrib)
    metadata['spatial_units'] = spatial_units
    metadata['time_units'] = time_units

    return spots, metadata


path = r"C:\Users\modri\Desktop\img_sq.xml"

# data, metadata = load_trackmate_full_xml(path)
# data.sort_values(by=['TRACK_ID', 'ID'], inplace=True)
# data

load_trackmate_full_xml(path)

(       TRACK_ID     ID     name  STD_INTENSITY_CH1   QUALITY  POSITION_T  \
 0         149.0   6438   ID6438          30.981987  2.949416         0.0   
 1         153.0   6439   ID6439          36.344565  4.086267         0.0   
 2         154.0   6440   ID6440          36.351857  3.226291         0.0   
 3         155.0   6441   ID6441          45.190139  4.653025         0.0   
 4         156.0   6442   ID6442          22.863086  2.019711         0.0   
 ...         ...    ...      ...                ...       ...         ...   
 53439     141.0  58065  ID58065          22.538233  3.169630      8010.0   
 53440    1529.0  58066  ID58066           9.563805  1.123792      8010.0   
 53441    1577.0  58067  ID58067          11.955960  0.955964      8010.0   
 53442     148.0  58068  ID58068          21.213836  2.162665      8010.0   
 53443     721.0  58069  ID58069          29.199048  3.240623      8010.0   
 
        MIN_INTENSITY_CH1  TOTAL_INTENSITY_CH1  CONTRAST_CH1   SNR_CH1  FR

In [4]:
path2 = r"C:\Users\modri\Desktop\position_0003440_allspots.csv"

In [115]:
import pandas as pd
import csv

def load_trackmate_csv(path, units_row_index=2, skiprows=4):

    column_names = pd.read_csv(path, nrows=0).columns.tolist()
    if units_row_index is not None:
        units_row = pd.read_csv(path, skiprows=units_row_index, nrows=1).iloc[0].tolist()

    print(f"Machine names: {column_names}")
    print(f"Units row: {units_row}")

    units = {}
    for col, u in zip(column_names, units_row):
        if isinstance(u, str):
            u = u.strip()
            if u.startswith('(') and u.endswith(')'):
                u = u[1:-1]
        else:
            u = None
        units[col] = u

    df = pd.read_csv(path, names=column_names, skiprows=skiprows, low_memory=False)

    return df, units


data, units = load_trackmate_csv(path2)

Machine names: ['LABEL', 'ID', 'TRACK_ID', 'QUALITY', 'POSITION_X', 'POSITION_Y', 'POSITION_Z', 'POSITION_T', 'FRAME', 'RADIUS', 'VISIBILITY', 'MANUAL_SPOT_COLOR', 'MEAN_INTENSITY_CH1', 'MEDIAN_INTENSITY_CH1', 'MIN_INTENSITY_CH1', 'MAX_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'STD_INTENSITY_CH1', 'EXTRACK_P_STUCK', 'EXTRACK_P_DIFFUSIVE', 'CONTRAST_CH1', 'SNR_CH1']
Units row: [np.float64(nan), np.float64(nan), np.float64(nan), '(quality)', '(micron)', '(micron)', '(micron)', '(sec)', np.float64(nan), '(micron)', np.float64(nan), np.float64(nan), '(counts)', '(counts)', '(counts)', '(counts)', '(counts)', '(counts)', np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan)]


In [116]:
load_trackmate_csv(path2)

Machine names: ['LABEL', 'ID', 'TRACK_ID', 'QUALITY', 'POSITION_X', 'POSITION_Y', 'POSITION_Z', 'POSITION_T', 'FRAME', 'RADIUS', 'VISIBILITY', 'MANUAL_SPOT_COLOR', 'MEAN_INTENSITY_CH1', 'MEDIAN_INTENSITY_CH1', 'MIN_INTENSITY_CH1', 'MAX_INTENSITY_CH1', 'TOTAL_INTENSITY_CH1', 'STD_INTENSITY_CH1', 'EXTRACK_P_STUCK', 'EXTRACK_P_DIFFUSIVE', 'CONTRAST_CH1', 'SNR_CH1']
Units row: [np.float64(nan), np.float64(nan), np.float64(nan), '(quality)', '(micron)', '(micron)', '(micron)', '(sec)', np.float64(nan), '(micron)', np.float64(nan), np.float64(nan), '(counts)', '(counts)', '(counts)', '(counts)', '(counts)', '(counts)', np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan)]


(         LABEL     ID  TRACK_ID   QUALITY  POSITION_X  POSITION_Y  POSITION_Z  \
 0       ID6438   6438     149.0  2.949416  107.037336    0.000000         0.0   
 1       ID6439   6439     153.0  4.086267  618.601983    0.000000         0.0   
 2       ID6440   6440     154.0  3.226291  669.536991    0.000000         0.0   
 3       ID6441   6441     155.0  4.653025  720.471999    0.000000         0.0   
 4       ID6442   6442     156.0  2.019711  971.456097    0.000000         0.0   
 ...        ...    ...       ...       ...         ...         ...         ...   
 53439  ID58065  58065     141.0  3.169630  668.458968  875.951429         0.0   
 53440  ID58066  58066    1529.0  1.123792  830.026108  879.125552         0.0   
 53441  ID58067  58067    1577.0  0.955964  487.942614  885.088040         0.0   
 53442  ID58068  58068     148.0  2.162665  611.220098  885.088040         0.0   
 53443  ID58069  58069     721.0  3.240623  705.708229  885.088040         0.0   
 
        POSITI

In [54]:
from typing import Dict, Optional
from warnings import warn

class InputWarning(Warning): ...

class InputMetadata:
    """
    Input metadata container:
    -------------------------

    A dictionary mapping metadata to file names:

        {
            "file1.csv": {
                "spatial_units": str,
                "time_units": str, 
                "time_interval": float,
                "n_frames": int,
            },
            "file2.csv": ...,
        }
    """

    def __init__(self, input_metadata: Optional[Dict[str, dict]] = None):
        self.input_metadata: Dict[str, dict] = input_metadata or {}

    def update(self, metadata: Dict[str, dict]):
        self.input_metadata.update(metadata)

    def get(self, file_name: str = None) -> Optional[dict]:
        if file_name is None:
            return self.input_metadata
        return self.input_metadata.get(file_name)

    def check(self) -> None:
        all_time_units = set()
        all_spatial_units = set()
        all_n_frames = set()
        all_time_intervals = set()

        for file_name, metadata in self.input_metadata.items(): 
            all_time_units.add(metadata.get("time_units"))
            all_spatial_units.add(metadata.get("spatial_units"))
            all_n_frames.add(metadata.get("n_frames"))
            all_time_intervals.add(metadata.get("time_interval"))

        if len(all_time_units) > 1:
            warn(
                f"Conflicting time units found in input metadata: {all_time_units}. "
                "Please ensure that all input files use the same time units.",
                InputWarning,
                stacklevel=2
            )
        if len(all_spatial_units) > 1:
            warn(
                f"Conflicting spatial units found in input metadata: {all_spatial_units}. "
                "Please ensure that all input files use the same spatial units.",
                InputWarning,
                stacklevel=2
            )

        if len(all_n_frames) > 1:
            warn(
                f"Conflicting number of frames found in input metadata: {all_n_frames}. "
                "Please ensure that all input files have the same number of frames.",
                InputWarning,
                stacklevel=2
            )

        if len(all_time_intervals) > 1:
            warn(
                f"Conflicting time intervals found in input metadata: {all_time_intervals}. "
                "Please ensure that all input files use the same time interval.",
                InputWarning,
                stacklevel=2
            )

input_metadata = InputMetadata()

In [55]:
input_metadata.update({
    "file1.csv": {
        "spatial_units": units['POSITION_X'],  # example value
        "time_units": units['POSITION_T'],
        "time_interval": 1.0,  # example value
        "n_frames": 100, 
    },
    "file2.csv": {
        "spatial_units": 'min',  # example value
        "time_units": units['POSITION_T'],
        "time_interval": 1.0,  # example value
        "n_frames": 200, 
    },
})

input_metadata.check()


C:\Users\modri\AppData\Local\Temp\ipykernel_147780\109137151.py:16: InputWarning: Conflicting spatial units found in input metadata: {'min', 'micron'}. Please ensure that all input files use the same spatial units.
  input_metadata.check()
C:\Users\modri\AppData\Local\Temp\ipykernel_147780\109137151.py:16: InputWarning: Conflicting number of frames found in input metadata: {200, 100}. Please ensure that all input files have the same number of frames.
  input_metadata.check()
